In [1]:
!pip install torch transformers

In [3]:
import json
import os
import torch
CLASSES = {
    "Adware": 0,
    "Backdoor": 1,
    "Botnet": 2,
    "CGI": 3,
    "Code-execution": 4,
    "DDos": 5,
    "Dir-Traversal": 6,
    "Dos": 7,
    "Info-Disclosure": 8,
    "Injection": 9,
    "Other": 10,
    "Overflow": 11,
    "Ransomware": 12,
    "Remote-file-Inclusion": 13,
    "Scanner": 14,
    "Spyware": 15,
    "Trojan": 16,
    "Virus": 17,
    "Webshell": 18,
    "Worm": 19,
    "XSS": 20
}
INV_CLASSES = {v: k for k, v in CLASSES.items()}
CONCEPTS= ["ip", "injection"]
CLASSES_TO_EXAMINE = ["Adware", "Scanner", "Spyware", "Trojan", "XSS", "Remote-file-Inclusion", "Overflow", "Injection", "Info-Disclosure", "Dir-Traversal", "Code-execution", "CGI", "Ransomware", "Botnet", "Backdoor"]
MODEL_NAME = "./codebert-base-mlm"


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [4]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import numpy as np
import random

In [25]:
# Estrazione delle feature
f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
dataset = [json.loads(line) for line in f.readlines()]
f.close()

f = open("./data/packet_inspection/anomalous_packets.jsonl", "r")
val_test_dataset = [json.loads(line) for line in f.readlines()]
f.close()

dataset = random.sample(val_test_dataset, 1000)

val_dataset = random.sample([d for d in val_test_dataset if d not in dataset], 100)
test_dataset = random.sample([d for d in val_test_dataset if d not in val_dataset and d not in dataset], 100)

f = open("./data/packet_inspection/anomalous_packets.jsonl", "r")
val_test_dataset = [json.loads(line) for line in f.readlines()]
f.close()

train_dataset = random.sample(val_test_dataset, 1000)

val_dataset = random.sample(ai_dataset, 100)
test_dataset = random.sample([d for d in ai_dataset if d not in val_dataset], 100)

tokenized_inputs = tokenizer([s["text"] for s in train_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**tokenized_inputs)
    hidden_states = outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    activations = [h.numpy() for h in hidden_states]  # Converti in numpy array
    
val_tokenized_inputs = tokenizer([s["text"] for s in val_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    val_outputs = model(**val_tokenized_inputs)
    val_hidden_states = val_outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    val_activations = [h.numpy() for h in val_hidden_states]
    
test_tokenized_inputs = tokenizer([s["text"] for s in test_dataset], padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    test_outputs = model(**test_tokenized_inputs)
    test_hidden_states = test_outputs.hidden_states
    # Prendi l'ultimo layer nascosto
    test_activations = [h.numpy() for h in test_hidden_states]

Done training inputs
Done validating inputs
Done testing inputs


In [7]:
class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
def calculate_sae_loss(x, W_enc, b_enc, W_dec, b_dec, lambd):
    """
    Calcola la loss function per uno Sparse AutoEncoder come specificato nell'immagine.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).
        b_dec (torch.Tensor): Bias del decoder, di forma (D,).
        lambd (float): Parametro di regolarizzazione.

    Returns:
        torch.Tensor: Il valore scalare della loss.
    """
    # 1. Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    # Il prodotto matriciale (matmul) deve tenere conto delle dimensioni
    # Il documento indica W_enc come (F, D), quindi lo usiamo direttamente
    # per matmul(W_enc, x.T) o usiamo x con matmul(x, W_enc.T)
    # Calcola le attivazioni delle feature: per ogni feature i, W_enc[i] @ x
    # x: (batch_size, 768), W_enc: (6144, 768)
    # Risultato: (batch_size, 6144)
    feature_activations = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    print(feature_activations.shape)

    # 2. Ricostruzione dell'input (x_hat = W_dec * f(x) + b_dec)
    # W_dec è di forma (D, F) e feature_activations di forma (batch_size, F)
    x_hat = torch.matmul(feature_activations, W_dec.T) + b_dec
    
    # 3. Calcolo del termine di ricostruzione (L2 norm)
    # ||x - x_hat||_2^2
    reconstruction_loss = torch.mean(torch.sum((x - x_hat)**2, dim=1))
    
    # 4. Calcolo del termine di penalizzazione (L1 norm)
    # lambda * sum(f_i(x) * ||W_dec_i||_2)
    # La tua immagine mostra un L2 norm sui pesi del decoder,
    # ma una regolarizzazione L1 sulle attivazioni.
    # Spieghiamola così: la somma dei valori assoluti delle attivazioni moltiplicata per il peso
    # e una lambda.
    # W_dec è (D,F), quindi W_dec_i (cioè W_dec[:,i]) è un vettore colonna D-dimensionale.
    # La norma ||W_dec_i||_2 è la norma L2 di questa colonna.
    # Lo sum sulle colonne ci dà un vettore (F,) con la norma L2 di ogni colonna di W_dec.
    W_dec_norms = torch.norm(W_dec, p=2, dim=0)
    
    # Calcola la penalizzazione: somma del prodotto delle attivazioni e delle norme
    l1_penalty = torch.mean(torch.matmul(feature_activations, W_dec_norms))

    # 5. Calcolo della loss totale
    total_loss = reconstruction_loss + lambd * l1_penalty

    return total_loss

def get_feature_directions(W_dec):
    """
    Calcola i vettori di direzione delle feature a partire dalla matrice dei pesi del decoder.

    Args:
        W_dec (torch.Tensor): Pesi del decoder di forma (D, F), dove
                              D è la dimensione residua e F la dimensione delle feature.

    Returns:
        torch.Tensor: I vettori delle feature (direzioni normalizzate), di forma (D, F).
    """
    # Calcola la norma L2 di ogni colonna (dim=0) della matrice W_dec.
    # Aggiunge 1e-8 per evitare divisioni per zero.
    norms = torch.norm(W_dec, p=2, dim=0, keepdim=True)
    
    # Normalizza ogni colonna (vettore di feature) dividendo per la sua norma.
    feature_directions = W_dec / (norms + 1e-8)
    
    return feature_directions

def get_feature_activations(x, W_enc, b_enc):
    """
    Calcola le attivazioni delle feature per un dato input x.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).

    Returns:
        torch.Tensor: Le attivazioni delle feature, di forma (batch_size, F).
    """
    # Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    feature_activations = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    
    return feature_activations

In [9]:
from itertools import product
from tqdm import tqdm

SKIP = True
os.makedirs("saved_models", exist_ok=True)

input_dim = 768  # Dimension of RoBERTa embeddings
hidden_dim = 768*8  # Dimension of the hidden layer in SAE

beta = 3.0      # Peso della penalizzazione
sae = SAE(input_dim, hidden_dim)

criterion = nn.MSELoss(reduction='sum')
optimizer = optim.Adam(sae.parameters(), lr=5e-5)

# Random data for demonstration
# Genera un dataset casuale di numeri
num_samples = 100  # Ridotto il numero di campioni per test più veloci
np.random.seed(42)

if not SKIP:
    # Definisci gli iperparametri da testare
    betas = np.arange(1.0, 5.5, 0.5)  # Da 1.0 a 5.0 con passo 0.5
    lrs = np.arange(1e-5, 1.1e-4, 1e-5)  # Da 1e-5 a 1e-4 con passo 1e-5
    best_loss = float('inf')
    best_params = None
    
    previous_results = []
    with open("./saved_models/training_results.json", "r") as f:
        for l in f.readlines():
            previous_results.append(json.loads(l))
    results = []
    best_models = {}
    
    for beta, lr in product(betas, lrs):
        for i, inputs in enumerate(activations):
            # Check if combination already performed
            found = False
            for trial in previous_results:
                if trial['layer'] == i and trial['hidden_dim'] == hidden_dim and trial["beta"] == beta and trial["lr"] == lr:
                    found = True
                    break
            # Skip training if already performed        
            if found:
                break
            X_train = inputs
            train_dataset = data.TensorDataset(torch.from_numpy(X_train))
            train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)  
            sae = SAE(input_dim, hidden_dim)
            optimizer = optim.Adam(sae.parameters(), lr=lr)
            epochs = 50
            for epoch in tqdm(range(epochs), desc=f"Layer {i} beta={beta} lr={lr}"):
                epoch_loss = 0
                for batch_idx, (data_batch,) in enumerate(train_loader):
                    reconstructed_data, encoded_activations = sae(data_batch)
                    encoder_weights = sae.encoder[0].weight
                    decoder_weights = sae.decoder[0].weight
                    loss = calculate_sae_loss(
                        data_batch, 
                        encoder_weights, 
                        sae.encoder[0].bias, 
                        decoder_weights, 
                        sae.decoder[0].bias, 
                        lambd=beta
                    )
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                avg_loss = epoch_loss / len(X_train)
            print(f"Layer: {i}\t hidden_dim={hidden_dim}, beta={beta}, lr={lr}, avg_loss={avg_loss:.4f}")
            
            # Salva ogni risultato
            x = {"layer": i, "hidden_dim": hidden_dim, "beta": beta, "lr": lr, "avg_loss": avg_loss}
            with open("saved_models/training_results.json", "a") as f:
                json.dump(x, f)
            results.append(x)
            
            # Salva il modello migliore per ogni layer
            if i not in best_models or avg_loss < best_models[i]["loss"]:
                best_models[i] = {
                    "model_state_dict": sae.state_dict(),
                    "loss": avg_loss,
                    "hidden_dim": hidden_dim,
                    "beta": beta,
                    "lr": lr
                }
    
    # Salva i modelli migliori per ogni layer
    
    for i, info in best_models.items():
        torch.save(info["model_state_dict"], f"saved_models/best_sae_layer_{i}.pt")
    
    print("Best models saved in 'saved_models/' and training details in 'training_results.json'")

Layer 0 beta=2.5 lr=1e-05:   2%|▏         | 1/50 [00:13<10:50, 13.27s/it]


KeyboardInterrupt: 

In [73]:
# Validazione di singolo modello
USE_PRETRAINED = True
hidden_dim = 768*32
layer = 10  # Layer da validare
beta = 9.0

sae = SAE(input_dim, hidden_dim)

X_train = activations[layer]
train_dataset = data.TensorDataset(torch.from_numpy(X_train))
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)

optimizer = optim.Adam(sae.parameters(), lr=5e-5)
epochs = 200
patience = 10
best_loss = float('inf')
epochs_no_improve = 0
best_state_dict = None

val_X = val_activations[layer]
val_dataset = data.TensorDataset(torch.from_numpy(val_X))
val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)

if USE_PRETRAINED:
    path = f"saved_models/best_sae_layer_{layer}.pt"
    if os.path.exists(path):
        sae.load_state_dict(torch.load(path))
        print(f"Modello pre-addestrato caricato da {path}")
        skip_training = True
    else:
        print(f"Modello non trovato in {path}, verrà addestrato un nuovo modello.")
        skip_training = False
else:
    skip_training = False

if skip_training:
    epochs = 0  # Salta il training loop
for epoch in range(epochs):
    epoch_loss = 0
    sae.train()
    for batch_idx, (data_batch,) in enumerate(train_loader):
        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        loss = calculate_sae_loss(
            data_batch,
            encoder_weights,
            sae.encoder[0].bias,
            decoder_weights,
            sae.decoder[0].bias,
            lambd=beta
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(X_train)

    # Validazione
    sae.eval()
    val_loss = 0
    with torch.no_grad():
        for val_batch, in val_loader:
            encoder_weights = sae.encoder[0].weight
            decoder_weights = sae.decoder[0].weight
            v_loss = calculate_sae_loss(
                val_batch,
                encoder_weights,
                sae.encoder[0].bias,
                decoder_weights,
                sae.decoder[0].bias,
                lambd=beta
            )
            val_loss += v_loss.item()
    avg_val_loss = val_loss / len(val_X)

    print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, val_loss={avg_val_loss:.4f}")
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        best_state_dict = sae.state_dict()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Carica il miglior modello trovato e valutalo sul test set
test_X = test_activations[layer]
test_dataset = data.TensorDataset(torch.from_numpy(test_X))
test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False)

sae.eval()
test_loss = 0
with torch.no_grad():
    for test_batch, in test_loader:
        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        t_loss = calculate_sae_loss(
            test_batch,
            encoder_weights,
            sae.encoder[0].bias,
            decoder_weights,
            sae.decoder[0].bias,
            lambd=beta
        )
        test_loss += t_loss.item()
avg_test_loss = test_loss / len(test_X)
print(f"Test loss: {avg_test_loss:.4f}")


if best_state_dict is not None:
    sae.load_state_dict(best_state_dict)
    print(f"Best SAE model loaded with avg_loss={best_loss:.4f}")

Epoch 1: avg_loss=78.4087, val_loss=72.7853
Epoch 2: avg_loss=25.5671, val_loss=43.6607
Epoch 3: avg_loss=15.7657, val_loss=33.3181
Epoch 4: avg_loss=12.9724, val_loss=27.5845
Epoch 5: avg_loss=11.3477, val_loss=23.6616
Epoch 6: avg_loss=10.1656, val_loss=20.7408
Epoch 7: avg_loss=9.2161, val_loss=18.4627
Epoch 8: avg_loss=8.5552, val_loss=16.7620
Epoch 9: avg_loss=8.1080, val_loss=15.4980
Epoch 10: avg_loss=7.7800, val_loss=14.5720
Epoch 11: avg_loss=7.5847, val_loss=13.8904
Epoch 12: avg_loss=7.4430, val_loss=13.3690
Epoch 13: avg_loss=7.3449, val_loss=12.9581
Epoch 14: avg_loss=7.2560, val_loss=12.6263
Epoch 15: avg_loss=7.1901, val_loss=12.3401
Epoch 16: avg_loss=7.1336, val_loss=12.0981
Epoch 17: avg_loss=7.0722, val_loss=11.8941
Epoch 18: avg_loss=7.0251, val_loss=11.7123
Epoch 19: avg_loss=6.9667, val_loss=11.5468
Epoch 20: avg_loss=6.9353, val_loss=11.4047
Epoch 21: avg_loss=6.8882, val_loss=11.2611
Epoch 22: avg_loss=6.8428, val_loss=11.1626
Epoch 23: avg_loss=6.8170, val_loss

In [74]:
# Estrazione delle feature di concetto

# Estrai le direzioni delle feature dal decoder del SAE
encoder_weights = sae.encoder[0].weight
decoder_weights = sae.decoder[0].weight

# print(decoder_weights.shape)
# print(encoder_weights.shape)

feature_directions = get_feature_directions(decoder_weights)

# Calcola le attivazioni delle feature sui dati di test
test_X_tensor = torch.from_numpy(test_X)
with torch.no_grad():
    f = torch.relu(torch.matmul(test_X_tensor, encoder_weights.T) + sae.encoder[0].bias)
    decoder_norms = torch.norm(decoder_weights, p=2, dim=0)
    # print("f", f.shape)
    # print("decoder", decoder_norms.shape)
    feature_activations = f * decoder_norms

# print(feature_activations.shape)

# Calcola la percentuale di feature attivate (diverse da zero) per ciascun esempio
nonzero_counts = (feature_activations != 0).sum(dim=2)  # shape: (100, 512)
# print(nonzero_counts)
flattened_nonzero_counts = nonzero_counts.flatten()
# print(flattened_nonzero_counts)
sparsity = flattened_nonzero_counts.float().mean().item()
print(f"Sparsità media (numero di feature attive per singolo token): {sparsity:.4f}")


# Analizza la sparsità delle attivazioni
mean_activations = feature_activations.mean(dim=(0,1))
live_features = (mean_activations!=0).float().mean().item()

print(f"Percerntuale di attivazione features (% di feature \"vive\"): {live_features:.4f}")

dead_features = (mean_activations == 0).sum().item()
print(f"Numero di dead features (mai attivate): {dead_features} su {len(mean_activations)}")


# Determina le feature più attive (concetti individuati)

top_features = torch.topk(mean_activations, k=10)
print("Top 10 feature (concetti) più attive:")
for idx, value in zip(top_features.indices.tolist(), top_features.values.tolist()):
    print(idx, value)
    print(f"Feature {idx}: attivazione media = {value:.4f}")

Sparsità media (numero di feature attive per singolo token): 1820.8953
Percerntuale di attivazione features (% di feature "vive"): 0.9976
Numero di dead features (mai attivate): 59 su 24576
Top 10 feature (concetti) più attive:
16825 0.37442442774772644
Feature 16825: attivazione media = 0.3744
19861 0.3740769326686859
Feature 19861: attivazione media = 0.3741
4851 0.3226894438266754
Feature 4851: attivazione media = 0.3227
4116 0.2801400125026703
Feature 4116: attivazione media = 0.2801
878 0.2421855330467224
Feature 878: attivazione media = 0.2422
16505 0.22599594295024872
Feature 16505: attivazione media = 0.2260
3758 0.21605955064296722
Feature 3758: attivazione media = 0.2161
9447 0.21492058038711548
Feature 9447: attivazione media = 0.2149
15329 0.1385774165391922
Feature 15329: attivazione media = 0.1386
24441 0.12577611207962036
Feature 24441: attivazione media = 0.1258


In [75]:
# Salva lo stato del modello SAE migliore trovato durante la validazione
if best_state_dict is not None:
    torch.save(
        best_state_dict,
        f"saved_models/best_sae_validated_beta{beta}_lr{optimizer.param_groups[0]['lr']}_dim{hidden_dim}.pt"
    )
    print(f"Modello SAE migliore salvato in 'saved_models/best_sae_validated_beta{beta}_lr{optimizer.param_groups[0]['lr']}_dim{hidden_dim}.pt'")

Modello SAE migliore salvato in 'saved_models/best_sae_validated_beta9.0_lr5e-05_dim24576.pt'


In [76]:
# Raggruppa i token che hanno le stesse feature attivate (non nulle) in feature_activations

# feature_activations shape: (100, 512, 6144)
# Per ogni token (sample, token), trova l'indice delle feature non nulle
groups = {}
for sample_idx in range(feature_activations.shape[0]):
    for token_idx in range(feature_activations.shape[1]):
        # Trova le feature attivate (non nulle) per questo token
        active_features = tuple(torch.nonzero(feature_activations[sample_idx, token_idx]).squeeze().tolist())
        # Usa la tupla degli indici come chiave
        if active_features not in groups:
            groups[active_features] = []
        groups[active_features].append((sample_idx, token_idx))

# groups ora contiene: chiave = tuple di feature attivate, valore = lista di (sample_idx, token_idx)
print(f"Numero di gruppi trovati: {len(groups)}")
for k, v in list(groups.items())[:5]:  # Mostra solo i primi 5 gruppi
    print(f"Features attivate: {k}\nToken indices: {v}\n")

Numero di gruppi trovati: 12892
Features attivate: (1, 2, 5, 6, 8, 9, 10, 15, 22, 27, 32, 35, 37, 39, 41, 43, 45, 48, 51, 54, 55, 58, 60, 64, 66, 67, 70, 76, 77, 78, 79, 84, 91, 93, 95, 96, 109, 119, 121, 125, 127, 128, 129, 134, 136, 137, 144, 145, 148, 154, 155, 159, 160, 162, 163, 164, 169, 175, 177, 179, 182, 184, 185, 186, 192, 194, 195, 203, 208, 211, 213, 214, 216, 217, 221, 224, 226, 227, 228, 229, 230, 236, 239, 240, 242, 251, 253, 257, 258, 263, 265, 267, 270, 274, 275, 280, 281, 282, 284, 285, 289, 292, 296, 298, 299, 302, 304, 306, 316, 317, 322, 325, 329, 330, 332, 342, 346, 347, 348, 350, 351, 358, 359, 363, 365, 367, 370, 371, 375, 376, 385, 394, 395, 397, 398, 402, 404, 405, 406, 408, 409, 415, 416, 417, 422, 425, 432, 435, 437, 441, 446, 448, 453, 455, 459, 461, 465, 466, 473, 474, 475, 479, 486, 494, 495, 498, 502, 505, 506, 512, 516, 529, 530, 531, 534, 542, 543, 544, 549, 550, 553, 555, 557, 561, 562, 563, 564, 572, 579, 582, 585, 593, 595, 598, 599, 600, 601, 603, 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
validations = random.sample([d for d in val_test_dataset if d not in train_dataset], 100)
inputs = [s["text"] for s in validations]
tokens_id = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
activations =  model(**tokens_id).hidden_states

num_samples, num_tokens, _ = activations[layer].shape
tokens_str = [tokenizer.tokenize(s, truncation=True,padding="max_length", max_length=num_tokens) for s in inputs]

print("Done validation inputs")

# parameters of sae
hidden_dim = 768*32
lr = 5e-5
beta = 9.0
layer = 10  # Layer da validare

sae = SAE(input_dim, hidden_dim)
path = f"saved_models/best_sae_validated_beta{beta}_lr{lr}_dim{hidden_dim}.pt"
sae.load_state_dict(torch.load(path))
print(f"Modello SAE caricato da {path}")

feature_activations = get_feature_activations(activations[layer], sae.encoder[0].weight, sae.encoder[0].bias)

In [ ]:
token_to_feature_activations = {}
num_samples, num_tokens, _ = activations[layer].shape
# print(activations[layer].shape)
# print(len(tokens_str[0]))
for sample_idx in range(num_samples):
    for token_idx in range(num_tokens):
        if token_idx == 0:
            token_to_feature_activations[(sample_idx, "CLS", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
        else:
            if tokens_id.attention_mask[sample_idx][token_idx] == 1:
                if tokens_str[sample_idx][token_idx-1] == "<pad>":
                    token_to_feature_activations[(sample_idx, "</s>", token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
                else:
                    token_to_feature_activations[(sample_idx, tokens_str[sample_idx][token_idx-1], token_idx)] = feature_activations[sample_idx, token_idx].detach().cpu().numpy()
            else:
                continue

print(token_to_feature_activations.keys())
first_key = next(iter(token_to_feature_activations))
print(token_to_feature_activations[first_key])

In [ ]:
from collections import defaultdict

# Crea un dizionario che, per ogni feature, raccoglie i token stringa che hanno attivazione non zero per quella feature
# Ora includi anche il token_idx nella tupla

feature_to_tokens = defaultdict(list)  # feature_idx -> lista di tuple (token stringa, sample_idx, token_idx, activation_value)

for (sample_idx, token_str, token_idx), activations in token_to_feature_activations.items():
    for feature_idx, activation_value in enumerate(activations):
        if activation_value != 0:
            feature_to_tokens[feature_idx].append((token_str, sample_idx, token_idx, activation_value))

# Ordina i token per ogni feature in base al valore di attivazione (decrescente)
for feature_idx in feature_to_tokens:
    feature_to_tokens[feature_idx] = sorted(
        feature_to_tokens[feature_idx],
        key=lambda x: abs(x[3]),  # ordina per valore assoluto dell'attivazione
        reverse=True
    )

# Ora feature_to_tokens[feature_idx] contiene l'insieme dei token stringa attivati per ogni feature
# Esempio: mostra i primi 5 feature e i loro token associati
for feature_idx in list(feature_to_tokens.keys())[:5]:
    print(f"Feature {feature_idx}: {list(feature_to_tokens[feature_idx])[:10]}")

NameError: name 'token_to_feature_activations' is not defined

In [ ]:
import json

# Funzione per serializzare i dati in formato richiesto
def serialize_feature_to_tokens(feature_to_tokens):
    result = {}
    for feature_idx, tokens in feature_to_tokens.items():
        result[str(feature_idx)] = [
            {
                "str": token_str,
                "sample_id": int(sample_idx),
                "token_id": int(token_idx),
                "activation": float(activation_value)
            }
            for token_str, sample_idx, token_idx, activation_value in tokens
        ]
    return result

serialized = serialize_feature_to_tokens(feature_to_tokens)

filename = f"feature_to_tokens_hidden{hidden_dim}_layer{layer}_beta{beta}_lr{lr}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(serialized, f, ensure_ascii=False, indent=2)
print(f"feature_to_tokens salvato in {filename}")

In [ ]:
# Validazione di singolo modello

hidden_dim = 768*8
layer = 11  # Layer da validare
beta = 3.0

sae = SAE(input_dim, hidden_dim)

X_train = activations[layer]
train_dataset = data.TensorDataset(torch.from_numpy(X_train))
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)

optimizer = optim.Adam(sae.parameters(), lr=5e-5)
epochs = 200
patience = 10
best_loss = float('inf')
epochs_no_improve = 0
best_state_dict = None

val_X = val_activations[layer]
val_dataset = data.TensorDataset(torch.from_numpy(val_X))
val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)
# Chiedi all'utente se caricare un modello già addestrato
USE_PRETRAINED = True  # Cambia a False per non usare modelli pre-addestrati

if USE_PRETRAINED:
    path = f"saved_models/best_sae_layer_{layer}.pt"
    if os.path.exists(path):
        sae.load_state_dict(torch.load(path))
        print(f"Modello pre-addestrato caricato da {path}")
        skip_training = True
    else:
        print(f"Modello non trovato in {path}, verrà addestrato un nuovo modello.")
        skip_training = False
else:
    skip_training = False

if skip_training:
    epochs = 0  # Salta il training loop

for epoch in range(epochs):
    epoch_loss = 0
    sae.train()
    for batch_idx, (data_batch,) in enumerate(train_loader):
        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        loss = calculate_sae_loss(
            data_batch,
            encoder_weights,
            sae.encoder[0].bias,
            decoder_weights,
            sae.decoder[0].bias,
            lambd=beta
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(X_train)

    # Validazione
    sae.eval()
    val_loss = 0
    with torch.no_grad():
        for val_batch, in val_loader:
            encoder_weights = sae.encoder[0].weight
            decoder_weights = sae.decoder[0].weight
            v_loss = calculate_sae_loss(
                val_batch,
                encoder_weights,
                sae.encoder[0].bias,
                decoder_weights,
                sae.decoder[0].bias,
                lambd=beta
            )
            val_loss += v_loss.item()
    avg_val_loss = val_loss / len(val_X)

    print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, val_loss={avg_val_loss:.4f}")
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        best_state_dict = sae.state_dict()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Carica il miglior modello trovato e valutalo sul test set
test_X = test_activations[layer]
test_dataset = data.TensorDataset(torch.from_numpy(test_X))
test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False)

sae.eval()
test_loss = 0
with torch.no_grad():
    for test_batch, in test_loader:
        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        t_loss = calculate_sae_loss(
            test_batch,
            encoder_weights,
            sae.encoder[0].bias,
            decoder_weights,
            sae.decoder[0].bias,
            lambd=beta
        )
        test_loss += t_loss.item()
avg_test_loss = test_loss / len(test_X)
print(f"Test loss: {avg_test_loss:.4f}")


if best_state_dict is not None:
    sae.load_state_dict(best_state_dict)
    print(f"Best SAE model loaded with avg_loss={best_loss:.4f}")

In [ ]:

# Estrazione delle feature di concetto

# Estrai le direzioni delle feature dal decoder del SAE
encoder_weights = sae.encoder[0].weight
decoder_weights = sae.decoder[0].weight

feature_directions = get_feature_directions(decoder_weights)

# Calcola le attivazioni delle feature sui dati di test
test_X_tensor = torch.from_numpy(test_X)
with torch.no_grad():
    f = torch.relu(torch.matmul(test_X_tensor, encoder_weights.T) + sae.encoder[0].bias)
    decoder_norms = torch.norm(decoder_weights, p=2, dim=0)
    feature_activations = f * decoder_norms

# Analizza la sparsità delle attivazioni
# Calcola la percentuale di feature attivate (diverse da zero) per ciascun esempio
nonzero_counts = (feature_activations != 0).sum(dim=2)  # shape: (100, 512)
# Concatena nonzero_counts in un array di lunghezza 51200 e calcola la media
flattened_nonzero_counts = nonzero_counts.flatten()
mean_percent_nonzero = flattened_nonzero_counts.float().mean().item()
print(f"Percentuale media di feature attivate (non nulle) per esempio: {mean_percent_nonzero:.4f}")
sparsity = (feature_activations == 0).float().mean().item()
print(f"Sparsità media delle attivazioni delle feature: {sparsity:.4f}")

# Analizza la quantità di dead features (feature mai attivate)
dead_features = (feature_activations.sum(dim=0) == 0).sum().item()
print(f"Numero di dead features (mai attivate): {dead_features} su {feature_activations.shape[1]}")

# Determina le feature più attive (concetti individuati)
mean_activations = feature_activations.mean(dim=0)
top_features = torch.topk(mean_activations, k=10)
print("Top 10 feature (concetti) più attive:")
for idx, value in zip(top_features.indices.tolist(), top_features.values.tolist()):
    print(f"Feature {idx}: attivazione media = {value:.4f}")

In [ ]:
# Salva lo stato del modello SAE migliore trovato durante la validazione
if best_state_dict is not None:
    torch.save(
        best_state_dict,
        f"saved_models/best_sae_validated_beta{beta}_lr{optimizer.param_groups[0]['lr']}_dim{hidden_dim}.pt"
    )
    print("Modello SAE migliore salvato in 'saved_models/best_sae_validated.pt'")

In [ ]:
# Raggruppa i token che hanno le stesse feature attivate (non nulle) in feature_activations

# feature_activations shape: (100, 512, 6144)
# Per ogni token (sample, token), trova l'indice delle feature non nulle
groups = {}
for sample_idx in range(feature_activations.shape[0]):
    for token_idx in range(feature_activations.shape[1]):
        # Trova le feature attivate (non nulle) per questo token
        active_features = tuple(torch.nonzero(feature_activations[sample_idx, token_idx]).squeeze().tolist())
        # Usa la tupla degli indici come chiave
        if active_features not in groups:
            groups[active_features] = []
        groups[active_features].append((sample_idx, token_idx))

# groups ora contiene: chiave = tuple di feature attivate, valore = lista di (sample_idx, token_idx)
print(f"Numero di gruppi trovati: {len(groups)}")
for k, v in list(groups.items())[:5]:  # Mostra solo i primi 5 gruppi
    print(f"Features attivate: {k}\nToken indices: {v}\n")

In [ ]:
import json

# Assumiamo che feature_activations abbia shape (num_samples, num_tokens, hidden_dim)
# e tokens_str sia una lista di liste di token stringa (senza CLS, con padding "<pad>")

output = []
num_samples, num_tokens, hidden_dim = feature_activations.shape

for sample_idx in tqdm(range(num_samples)):
    tokens_json = []
    # Primo token: CLS
    nonzero = feature_activations[sample_idx, 0].nonzero().squeeze().tolist()
    if isinstance(nonzero, int):
        nonzero = [nonzero]
    activations = [
        (int(i), float(feature_activations[sample_idx, 0, i].item()))
        for i in nonzero
        if feature_activations[sample_idx, 0, i] != 0
    ]
    tokens_json.append({
        "token_idx": 0,
        "token_str": "CLS",
        "activations": activations
    })
    # Token successivi
    for token_idx, token in enumerate(tokens_str[sample_idx]):
        fa_idx = token_idx + 1
        if fa_idx >= num_tokens:
            break
        if token == "<pad>":
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": "</s>",
                "activations": activations
            })
            break  # Stop at first <pad>
        else:
            nonzero = feature_activations[sample_idx, fa_idx].nonzero().squeeze().tolist()
            if isinstance(nonzero, int):
                nonzero = [nonzero]
            activations = [
                (int(i), float(feature_activations[sample_idx, fa_idx, i].item()))
                for i in nonzero
                if feature_activations[sample_idx, fa_idx, i] != 0
            ]
            tokens_json.append({
                "token_idx": fa_idx,
                "token_str": token,
                "activations": activations
            })
    output.append({
        "sample_index": sample_idx,
        "tokens": tokens_json
    })
print(f"Output JSON creato con {len(output)} samples.")
print(json.dumps(output[0], ensure_ascii=False, indent=2))
# Se vuoi salvarlo in un file:
# with open("tokens_with_activations_sparse.json", "w") as f:
#     json.dump(output, f, ensure_ascii=False, indent=2)

NameError: name 'feature_activations' is not defined